<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/04_legal_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4: Legal-BERT
Training Legal-BERT to detect the 8 types of unfair clauses, comparing class weighting, the extractive summaries and Legal-BERT against regular BERT.

## 1. Loading the data and the summaries
Loads UNFAIR-ToS and the extractive summaries saved in Drive. Each sentence gets a second version of its text: the summary if it was a long sentence, or the original text otherwise. Keeping both versions lets us compare training with and without the summaries.

In [1]:
!pip install -q datasets transformers accelerate

from datasets import load_dataset
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
ds = load_dataset("coastalcph/lex_glue", "unfair_tos")

# Load the saved summaries and keep the ones that worked
summ = pd.read_csv("/content/drive/MyDrive/fine_print_detector/long_sentence_summaries.csv")
summ = summ[summ["summary"].notna()]
lookup = {(s, int(i)): t for s, i, t in zip(summ["split"], summ["index"], summ["summary"])}

# Add a second text column: the summary if there is one, otherwise the original
for split in ["train", "validation", "test"]:
    ds[split] = ds[split].map(
        lambda ex, idx: {"text_summ": lookup.get((split, idx), ex["text"])},
        with_indices=True,
    )
    replaced = sum(1 for a, b in zip(ds[split]["text"], ds[split]["text_summ"]) if a != b)
    print(f"{split}: {replaced} sentences replaced by their summary")

# Show one example
i = int(summ[summ["split"] == "train"]["index"].iloc[0])
print("\nORIGINAL:", ds["train"][i]["text"][:200], "...")
print("SUMMARY: ", ds["train"][i]["text_summ"][:200], "...")

Mounted at /content/drive


README.md:   0%|          | 0.00/34.1k [00:01<?, ?B/s]

unfair_tos/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  501kB            

unfair_tos/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

unfair_tos/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147kB            

...

train: 70 sentences replaced by their summary


Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

validation: 39 sentences replaced by their summary


Map:   0%|          | 0/1607 [00:00<?, ? examples/s]

test: 17 sentences replaced by their summary

ORIGINAL: to the fullest extent permitted by applicable law , in no event will tinder , its affiliates , employees , licensors or service providers be liable for any indirect , consequential , exemplary , incid ...
SUMMARY:  to the fullest extent permitted by applicable law ... in no event will tinder , its affiliates , employees , licensors or service providers be liable ... indirect , consequential , exemplary , inciden ...


126 long sentences were replaced by their extractive summary (70 train, 39 validation, 17 test). The original text is kept in a separate column, so the model can be trained with or without the summaries.

## 2. Labels and tokenizing
Turns each label into 8 yes/no boxes and tokenizes the text with Legal-BERT's tokenizer (128 tokens). Two versions are made: one with the summaries and one with the original text, for the comparison in run C.

In [2]:
from transformers import AutoTokenizer

names = ds["train"].features["labels"].feature.names
MODEL = "nlpaueb/legal-bert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

# Labels as 8 yes/no boxes (decimals, as the training math needs)
def to_checklist(example):
    boxes = [0.0] * 8
    for l in example["labels"]:
        boxes[l] = 1.0
    return {"label_boxes": boxes}

ds = ds.map(to_checklist)

def make_tokenized(tokenizer, text_column):
    """Tokenize one text column to 128 tokens and attach the label boxes."""
    def tok_fn(batch):
        enc = tokenizer(batch[text_column], truncation=True, padding="max_length", max_length=128)
        enc["labels"] = batch["label_boxes"]
        return enc
    out = ds.map(tok_fn, batched=True, remove_columns=ds["train"].column_names)
    out.set_format("torch")
    return out

data_summ = make_tokenized(tok, "text_summ")     # with summaries (runs A, B)
data_orig = make_tokenized(tok, "text")          # original text (run C)

# Checks
print("Total ticked boxes in train:", int(sum(sum(row) for row in ds["train"]["label_boxes"])))
for col in ["text", "text_summ"]:
    over = sum(1 for t in ds["train"][col] if len(tok(t)["input_ids"]) > 128)
    print(f"Train sentences over 128 tokens using '{col}': {over}")

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

Map:   0%|          | 0/1607 [00:00<?, ? examples/s]

Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

Total ticked boxes in train: 702
Train sentences over 128 tokens using 'text': 71
Train sentences over 128 tokens using 'text_summ': 1


After tokenizing with Legal-BERT’s tokenizer, 71 training sentences exceed 128 tokens using the original text, but only 1 using the extractive summaries.

## 3. Class weights
Gives rare unfair types more importance during training. For each type, the weight is the square root of (sentences without the type / sentences with it), so missing a rare type costs more than missing a common one, without being so extreme that the model over-flags.

In [3]:
import torch
import numpy as np

boxes = np.array(ds["train"]["label_boxes"])
counts = boxes.sum(axis=0)                          # how many sentences have each type
n = len(boxes)
raw = (n - counts) / counts                          # "no" per "yes"
pos_weight = torch.tensor(np.sqrt(raw), dtype=torch.float)   # softened with a square root

print(f"{'Type':<26}{'Count':>7}{'Raw ratio':>11}{'Weight':>9}")
for name, c, r, w in zip(names, counts, raw, pos_weight):
    print(f"{name:<26}{int(c):>7}{r:>11.1f}{w.item():>9.1f}")

Type                        Count  Raw ratio   Weight
Limitation of liability       191       28.0      5.3
Unilateral termination        139       38.8      6.2
Unilateral change             122       44.3      6.7
Content removal                73       74.8      8.6
Contract by using              76       71.8      8.5
Choice of law                  39      140.8     11.9
Jurisdiction                   34      161.7     12.7
Arbitration                    28      196.6     14.0


Class weights range from 5.3 (Limitation of liability, the most common type) to 14.0 (Arbitration, the rarest). The square root keeps the weights from becoming so large that the model over-flags rare types.

## 4. Setting up the model
Loads Legal-BERT with a new final layer of 8 outputs, one per unfair type. Each output gives its own yes/no score (multi-label), so one sentence can have several types. A fresh model is created for every training run.

In [4]:
from transformers import AutoModelForSequenceClassification

def make_model(model_name):
    """Load a BERT-type model with 8 independent yes/no outputs (one per unfair type)."""
    return AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=8,
        problem_type="multi_label_classification",       # 8 separate yes/no answers
        id2label={i: n for i, n in enumerate(names)},
        label2id={n: i for i, n in enumerate(names)},
    )

model = make_model(MODEL)
print(model.classifier)                                  # the new final layer: the 8 judges
print("Total parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Linear(in_features=768, out_features=8, bias=True)
Total parameters: 109,488,392


Legal-BERT (about 110 million parameters) was loaded with a new final layer of 8 outputs. The warnings confirm that only this new layer starts untrained, as expected.

## 5. Training settings and scoring
Sets up the training, following the LexGLUE paper: learning rate 3e-5, batch size 8, up to 20 epochs with early stopping. Scores use micro and macro F1 with the extra "none" label from LexGLUE. Class weights are added to the loss when a run uses them. The best epoch is chosen by validation macro F1, since the rare types matter most in this project.

In [3]:
from sklearn.metrics import f1_score
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, set_seed

def lexglue_scores(y_true, y_pred):
    """Micro and macro F1 the LexGLUE way: add a 9th 'none' box for sentences with no unfair type."""
    def add_none(y):
        y = np.asarray(y)
        none = (y.sum(axis=1) == 0).astype(int).reshape(-1, 1)
        return np.hstack([y, none])
    t, p = add_none(y_true), add_none(y_pred)
    return (f1_score(t, p, average="micro", zero_division=0) * 100,
            f1_score(t, p, average="macro", zero_division=0) * 100)

def compute_metrics(eval_pred):
    """Turn the model's scores into yes/no (above 0.5 = yes), then score them."""
    logits, labels = eval_pred
    preds = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)
    micro, macro = lexglue_scores(labels.astype(int), preds)
    return {"micro_f1": micro, "macro_f1": macro}

class WeightedTrainer(Trainer):
    """A Trainer that can add class weights to the loss (the 'penalty' for wrong answers)."""
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        pw = self.pos_weight.to(outputs.logits.device) if self.pos_weight is not None else None
        loss = torch.nn.functional.binary_cross_entropy_with_logits(
            outputs.logits, labels.float(), pos_weight=pw)
        return (loss, outputs) if return_outputs else loss

def run_experiment(name, model_name, data, use_weights, seed=42):
    """Train one version from scratch and return the trained model and its validation scores."""
    set_seed(seed)                                           # same randomness every run
    args = TrainingArguments(
        output_dir=f"/content/runs/{name}",
        learning_rate=3e-5,                                # LexGLUE
        per_device_train_batch_size=8,                     # LexGLUE
        per_device_eval_batch_size=32,
        num_train_epochs=20,                               # LexGLUE: up to 20, with early stopping
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=True,                                         # LexGLUE: mixed precision
        logging_strategy="epoch",
        report_to="none",
        seed=seed,
    )
    trainer = WeightedTrainer(
        model=make_model(model_name),
        args=args,
        train_dataset=data["train"],
        eval_dataset=data["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],   # stop after 3 epochs without improvement
        pos_weight=pos_weight if use_weights else None,
    )
    trainer.train()
    return trainer, trainer.evaluate()

print("Ready to train.")

Ready to train.




---





## 6. Run A: Legal-BERT with summaries
Trains Legal-BERT on the data with extractive summaries, using the LexGLUE settings and no class weights. This is the reference run that the other runs are compared against. The scores are saved to Drive after the run.

In [6]:
import os

trainer_A, val_A = run_experiment("A_legalbert_summ", MODEL, data_summ, use_weights=False)
print(f"\nRun A (validation): micro F1 = {val_A['eval_micro_f1']:.1f}, macro F1 = {val_A['eval_macro_f1']:.1f}")

# Save the scores to Drive, so they're safe even if Colab disconnects
results_path = "/content/drive/MyDrive/fine_print_detector/run_results.csv"
row = pd.DataFrame([{"run": "A", "model": "legal-bert", "summaries": True, "class_weights": False,
                     "val_micro_f1": val_A["eval_micro_f1"], "val_macro_f1": val_A["eval_macro_f1"]}])
row.to_csv(results_path, mode="a", header=not os.path.exists(results_path), index=False)
print("Scores saved to Drive.")

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.085787,0.034332,90.741957,21.106520
2,0.031823,0.027656,92.974393,63.297533
3,0.019109,0.025041,94.716157,75.490879
4,0.012662,0.029633,93.374019,71.903350
5,0.008820,0.030406,93.922893,73.865010
6,0.005981,0.030813,94.858388,76.161158
7,0.004766,0.030267,94.022688,73.460245
8,0.003804,0.029148,94.830971,76.899973
9,0.002503,0.034604,94.294425,75.870523
10,0.002964,0.033001,94.573981,74.822710


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Micro F1,Macro F1
0.001999,0.029148,11,94.830971,76.899973



Run A (validation): micro F1 = 94.8, macro F1 = 76.9
Scores saved to Drive.


Run A reached its best validation scores at epoch 8 (micro F1 94.8, macro F1 76.9) and stopped at epoch 11. Validation loss started rising after epoch 3 while training loss kept falling, a sign of overfitting, but macro F1 kept improving until epoch 8. Compared with the TF-IDF + SVM baseline (69.1 macro F1 on validation), Legal-BERT improves macro F1 by 7.8 points.



---



## 7. Run B: adding class weights
The same as run A, but with class weights, so missing a rare type costs more during training. Comparing A and B shows whether class weights help.

In [7]:
trainer_B, val_B = run_experiment("B_legalbert_summ_weights", MODEL, data_summ, use_weights=True)
print(f"\nRun B (validation): micro F1 = {val_B['eval_micro_f1']:.1f}, macro F1 = {val_B['eval_macro_f1']:.1f}")

row = pd.DataFrame([{"run": "B", "model": "legal-bert", "summaries": True, "class_weights": True,
                     "val_micro_f1": val_B["eval_micro_f1"], "val_macro_f1": val_B["eval_macro_f1"]}])
row.to_csv(results_path, mode="a", header=False, index=False)
print("Scores saved to Drive.")

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.256227,0.115642,90.301999,61.958053
2,0.112478,0.102550,93.428757,71.412131
3,0.069394,0.098917,93.495758,73.653583
4,0.044385,0.095682,92.466638,72.074549
5,0.030952,0.162735,95.094833,76.108846
6,0.019900,0.121003,93.739130,74.064844
7,0.018089,0.129853,93.217391,71.869708
8,0.012773,0.137197,94.345368,75.326380


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Micro F1,Macro F1
0.012773,0.162735,8,95.094833,76.108846



Run B (validation): micro F1 = 95.1, macro F1 = 76.1
Scores saved to Drive.


With class weights, the best validation scores were micro F1 95.1 and macro F1 76.1, at epoch 5. Compared with run A (76.9), class weights did not improve macro F1, although the model learned the rare types faster in the first epochs (macro F1 62.0 vs 21.1 after epoch 1). Because the difference is below one point, run A’s setup (no class weights) is used for runs C and D.



---



## 8. Run C: without the summaries
The same as run A, but trained on the original text, so long sentences are simply cut at 128 tokens. Comparing A and C shows whether the extractive summaries help.

In [8]:
trainer_C, val_C = run_experiment("C_legalbert_orig", MODEL, data_orig, use_weights=False)
print(f"\nRun C (validation): micro F1 = {val_C['eval_micro_f1']:.1f}, macro F1 = {val_C['eval_macro_f1']:.1f}")

row = pd.DataFrame([{"run": "C", "model": "legal-bert", "summaries": False, "class_weights": False,
                     "val_micro_f1": val_C["eval_micro_f1"], "val_macro_f1": val_C["eval_macro_f1"]}])
row.to_csv(results_path, mode="a", header=False, index=False)
print("Scores saved to Drive.")

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.089876,0.039808,90.391771,17.359376
2,0.036724,0.028184,93.912285,61.758998
3,0.023078,0.026328,93.948001,73.417277
4,0.014812,0.032126,91.876629,69.117574
5,0.010374,0.024182,94.879059,75.911142
6,0.007233,0.026939,95.310796,76.499303
7,0.005616,0.026612,94.966224,76.578900
8,0.003558,0.029289,94.330571,75.102170
9,0.003365,0.033462,93.891798,73.469925
10,0.003626,0.032637,94.938918,75.870786


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Micro F1,Macro F1
0.003626,0.026612,10,94.966224,76.578900



Run C (validation): micro F1 = 95.0, macro F1 = 76.6
Scores saved to Drive.


Without summaries, the best validation scores were micro F1 95.0 and macro F1 76.6 (epoch 7), almost the same as run A (94.8 / 76.9). Because only about 1% of sentences were summarized, the overall scores cannot show the effect clearly, so the comparison is repeated on the long sentences only, and with several seeds.



---



## 9. Run D: regular BERT
The same as run A, but with regular BERT (bert-base-uncased) instead of Legal-BERT. The text is tokenized again with BERT's own tokenizer. Comparing A and D shows whether Legal-BERT's legal pretraining helps.

In [9]:
BERT = "bert-base-uncased"
bert_tok = AutoTokenizer.from_pretrained(BERT)
data_summ_bert = make_tokenized(bert_tok, "text_summ")       # same text, BERT's dictionary

trainer_D, val_D = run_experiment("D_bert_summ", BERT, data_summ_bert, use_weights=False)
print(f"\nRun D (validation): micro F1 = {val_D['eval_micro_f1']:.1f}, macro F1 = {val_D['eval_macro_f1']:.1f}")

row = pd.DataFrame([{"run": "D", "model": "bert", "summaries": True, "class_weights": False,
                     "val_micro_f1": val_D["eval_micro_f1"], "val_macro_f1": val_D["eval_macro_f1"]}])
row.to_csv(results_path, mode="a", header=False, index=False)
print("Scores saved to Drive.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Micro F1,Macro F1
0.006391,0.025447,6,94.438386,75.951703



Run D (validation): micro F1 = 94.4, macro F1 = 76.0
Scores saved to Drive.


Regular BERT reached its best validation scores at epoch 3 (micro F1 94.4, macro F1 76.0), 0.9 macro F1 below Legal-BERT (run A). This matches the LexGLUE finding that Legal-BERT performs better, though the difference here is small.



---



## 10. Test scores and saving
Scores all four runs on the test set and saves the scores and predictions to Drive, so those runs count as the first seed of each setup. The best model (run A) is also saved to Drive for the demo. The best setup is still chosen using validation scores only.

In [10]:
save_dir = "/content/drive/MyDrive/fine_print_detector"
os.makedirs(f"{save_dir}/predictions", exist_ok=True)

runs = {
    "A": (trainer_A, data_summ),
    "B": (trainer_B, data_summ),
    "C": (trainer_C, data_orig),
    "D": (trainer_D, data_summ_bert),
}

rows = []
for name, (trainer, data) in runs.items():
    out = trainer.predict(data["test"])                               # predictions on the test set
    np.save(f"{save_dir}/predictions/test_logits_{name}_seed42.npy", out.predictions)
    rows.append({"run": name, "seed": 42,
                 "test_micro_f1": out.metrics["test_micro_f1"],
                 "test_macro_f1": out.metrics["test_macro_f1"]})
    print(f"Run {name} (test): micro F1 = {out.metrics['test_micro_f1']:.1f}, macro F1 = {out.metrics['test_macro_f1']:.1f}")

test_path = f"{save_dir}/test_results.csv"
pd.DataFrame(rows).to_csv(test_path, mode="a", header=not os.path.exists(test_path), index=False)

# Save the best model (run A) for the demo
trainer_A.save_model(f"{save_dir}/models/legal_bert_A")
tok.save_pretrained(f"{save_dir}/models/legal_bert_A")
print("\nTest scores, predictions and model A saved to Drive.")

Run A (test): micro F1 = 96.0, macro F1 = 83.8


Run B (test): micro F1 = 95.3, macro F1 = 79.3


Run C (test): micro F1 = 95.6, macro F1 = 81.9


Run D (test): micro F1 = 94.8, macro F1 = 80.5


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Test scores, predictions and model A saved to Drive.


In [11]:
for name, (trainer, _) in runs.items():
    if name != "A":
        trainer.save_model(f"{save_dir}/models/run_{name}")
print("Models B, C and D saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Models B, C and D saved.




---



## 11. Extra seeds
Trains every setup (A, B, C, D) two more times with different seeds (1 and 2), because a single run can be lucky or unlucky. After each run, its validation and test scores and its test predictions are saved to Drive. Together with seed 42, every setup then has 3 runs to average.

In [13]:
import gc, shutil

# Free the GPU memory used by tonight's runs (their results are already saved)
del trainer_B, trainer_C, trainer_D
gc.collect(); torch.cuda.empty_cache()

setups = [
    ("A", MODEL, data_summ, False),
    ("C", MODEL, data_orig, False),
    ("B", MODEL, data_summ, True),
    ("D", BERT, data_summ_bert, False),
]

for seed in [1, 2]:
    for name, model_name, data, use_weights in setups:
        run_name = f"{name}_seed{seed}"
        trainer, val = run_experiment(run_name, model_name, data, use_weights, seed=seed)
        out = trainer.predict(data["test"])

        # Save everything for this run
        np.save(f"{save_dir}/predictions/test_logits_{name}_seed{seed}.npy", out.predictions)
        pd.DataFrame([{"run": name, "seed": seed,
                       "val_micro_f1": val["eval_micro_f1"], "val_macro_f1": val["eval_macro_f1"]}]).to_csv(
            f"{save_dir}/run_results_seeds.csv", mode="a",
            header=not os.path.exists(f"{save_dir}/run_results_seeds.csv"), index=False)
        pd.DataFrame([{"run": name, "seed": seed,
                       "test_micro_f1": out.metrics["test_micro_f1"],
                       "test_macro_f1": out.metrics["test_macro_f1"]}]).to_csv(
            test_path, mode="a", header=False, index=False)

        print(f"✅ {run_name}: val macro {val['eval_macro_f1']:.1f} | test macro {out.metrics['test_macro_f1']:.1f}")

        # Clean up before the next run
        del trainer
        gc.collect(); torch.cuda.empty_cache()
        shutil.rmtree(f"/content/runs/{run_name}", ignore_errors=True)

print("\nAll extra runs done and saved.")

[transformers] You passed `num_labels=8` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.079759,0.035706,92.920932,46.270671
2,0.029645,0.026395,94.050992,73.982063
3,0.017534,0.026196,94.122769,72.254277
4,0.010523,0.026445,94.913774,74.340698
5,0.007829,0.027713,94.720768,76.903281
6,0.005622,0.027650,94.808028,76.493015
7,0.004184,0.031188,94.027899,74.215996
8,0.003098,0.029157,95.227718,77.904278
9,0.002869,0.035576,93.928183,74.194606
10,0.002951,0.035045,94.247884,74.396290


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Micro F1,Macro F1
0.003426,0.025063,8,95.154954,78.999918


✅ D_seed2: val macro 79.0 | test macro 79.2

All extra runs done and saved.


In [14]:
print("Prediction files:", len(os.listdir(f"{save_dir}/predictions")), "(should be 12)")
print("Test results rows:", len(pd.read_csv(f"{save_dir}/test_results.csv")), "(should be 12)")
print("Extra-seed validation rows:", len(pd.read_csv(f"{save_dir}/run_results_seeds.csv")), "(should be 8)")
print("Model A saved:", os.path.exists(f"{save_dir}/models/legal_bert_A"))

drive.flush_and_unmount()       # make sure everything is fully synced to Drive
print("Drive synced.")

Prediction files: 12 (should be 12)
Test results rows: 12 (should be 12)
Extra-seed validation rows: 8 (should be 8)
Model A saved: True
Drive synced.




---





---



## 12. Averages over 3 seeds
Combines the scores of all runs (seeds 42, 1 and 2) and shows the average and the spread (standard deviation) for each setup, on validation and test. The spread shows how much the score changes from luck alone.

In [4]:
save_dir = "/content/drive/MyDrive/fine_print_detector"

# Validation scores: seed 42 (first file) + seeds 1 and 2 (second file)
val_42 = pd.read_csv(f"{save_dir}/run_results.csv")[["run", "val_micro_f1", "val_macro_f1"]].assign(seed=42)
val_all = pd.concat([val_42, pd.read_csv(f"{save_dir}/run_results_seeds.csv")])
test_all = pd.read_csv(f"{save_dir}/test_results.csv")

names_map = {"A": "A: Legal-BERT + summaries", "B": "B: + class weights",
             "C": "C: no summaries", "D": "D: regular BERT"}

for title, df_, cols in [("VALIDATION", val_all, ["val_micro_f1", "val_macro_f1"]),
                         ("TEST", test_all, ["test_micro_f1", "test_macro_f1"])]:
    print(f"===== {title} (average ± spread over 3 seeds) =====")
    for run, g in df_.groupby("run"):
        mic, mac = g[cols[0]], g[cols[1]]
        print(f"{names_map[run]:<28} micro {mic.mean():.1f} ± {mic.std():.1f} | macro {mac.mean():.1f} ± {mac.std():.1f}")
    print()

===== VALIDATION (average ± spread over 3 seeds) =====
A: Legal-BERT + summaries    micro 94.6 ± 0.7 | macro 76.8 ± 1.2
B: + class weights           micro 95.2 ± 0.1 | macro 76.5 ± 0.5
C: no summaries              micro 94.7 ± 0.3 | macro 77.1 ± 0.7
D: regular BERT              micro 94.9 ± 0.4 | macro 77.5 ± 1.5

===== TEST (average ± spread over 3 seeds) =====
A: Legal-BERT + summaries    micro 95.8 ± 0.2 | macro 82.8 ± 1.3
B: + class weights           micro 95.8 ± 0.4 | macro 81.4 ± 2.3
C: no summaries              micro 95.2 ± 0.7 | macro 81.5 ± 2.8
D: regular BERT              micro 95.0 ± 0.2 | macro 79.1 ± 1.4



Averaged over three seeds, run A (Legal-BERT with summaries) achieved the best test scores: 95.8 ± 0.2 micro F1 and 82.8 ± 1.3 macro F1, matching LexGLUE’s reported Legal-BERT result (96.0 / 83.0). Legal-BERT clearly outperformed regular BERT on the test set (82.8 vs 79.1 macro F1, a gap larger than the seed variation). Class weights did not improve results. The summaries gave a small advantage (82.8 vs 81.5), but it is within the seed variation, so it is examined further on the long sentences.



---



## 13. The long sentences
Compares runs A (with summaries) and C (without) only on the 17 test sentences that were replaced by their summary, for all three seeds: how many sentences were fully correct, and how many unfair labels were caught, missed, or wrongly added.

In [4]:
import numpy as np
save_dir = "/content/drive/MyDrive/fine_print_detector"

y_true = np.array(ds["test"]["label_boxes"]).astype(int)
long_idx = [i for i, (a, b) in enumerate(zip(ds["test"]["text"], ds["test"]["text_summ"])) if a != b]
print("Long test sentences:", len(long_idx),
      "| of which unfair:", int((y_true[long_idx].sum(axis=1) > 0).sum()),
      "| unfair labels in total:", int(y_true[long_idx].sum()), "\n")

totals = {}
for run in ["A", "C"]:
    totals[run] = np.zeros(4, dtype=int)
    for seed in [42, 1, 2]:
        logits = np.load(f"{save_dir}/predictions/test_logits_{run}_seed{seed}.npy")
        pred = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)[long_idx]
        true = y_true[long_idx]
        stats = np.array([
            (pred == true).all(axis=1).sum(),          # fully correct sentences
            ((pred == 1) & (true == 1)).sum(),         # caught
            ((pred == 0) & (true == 1)).sum(),         # missed
            ((pred == 1) & (true == 0)).sum(),         # false alarms
        ])
        totals[run] += stats
        print(f"{run} seed {seed}: fully correct {stats[0]}/{len(long_idx)} | caught {stats[1]} | missed {stats[2]} | false alarms {stats[3]}")
    print()

for run, t in totals.items():
    label = "A (with summaries)" if run == "A" else "C (without)"
    print(f"{label:<20} over 3 seeds: fully correct {t[0]}/{3 * len(long_idx)} | caught {t[1]} | missed {t[2]} | false alarms {t[3]}")

Long test sentences: 17 | of which unfair: 6 | unfair labels in total: 6 

A seed 42: fully correct 17/17 | caught 6 | missed 0 | false alarms 0
A seed 1: fully correct 17/17 | caught 6 | missed 0 | false alarms 0
A seed 2: fully correct 17/17 | caught 6 | missed 0 | false alarms 0

C seed 42: fully correct 17/17 | caught 6 | missed 0 | false alarms 0
C seed 1: fully correct 17/17 | caught 6 | missed 0 | false alarms 0
C seed 2: fully correct 17/17 | caught 6 | missed 0 | false alarms 0

A (with summaries)   over 3 seeds: fully correct 51/51 | caught 18 | missed 0 | false alarms 0
C (without)          over 3 seeds: fully correct 51/51 | caught 18 | missed 0 | false alarms 0


On the 17 test sentences that were summarized, runs A and C were both fully correct in all three seeds (all 6 unfair labels caught, no false alarms). For these sentences, the key content already appeared within the first 128 tokens, so the summaries did not change test performance. Their value is in guaranteeing that important content is kept when it appears later in long clauses, as can happen in real Terms of Service.



---



## 14. Scores per type
Shows the test F1 score of the final model (run A, average over 3 seeds) for each unfair type, next to the TF-IDF + SVM baseline from Notebook 3 and the number of test examples of each type.

In [5]:
from sklearn.metrics import f1_score

# Per-type test F1 of the SVM baseline (from Notebook 3)
baseline = {"Limitation of liability": 77.6, "Unilateral termination": 66.7, "Unilateral change": 63.5,
            "Content removal": 76.9, "Contract by using": 80.0, "Choice of law": 91.7,
            "Jurisdiction": 74.1, "Arbitration": 60.0}

per_seed = []
for seed in [42, 1, 2]:
    logits = np.load(f"{save_dir}/predictions/test_logits_A_seed{seed}.npy")
    pred = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)
    per_seed.append(f1_score(y_true, pred, average=None, zero_division=0) * 100)
per_seed = np.array(per_seed)
counts = y_true.sum(axis=0)

print(f"{'Type':<26}{'Test ex.':>9}{'SVM':>8}{'Legal-BERT':>16}{'Change':>9}")
for i, name in enumerate(names):
    mean, std = per_seed[:, i].mean(), per_seed[:, i].std(ddof=1)
    print(f"{name:<26}{int(counts[i]):>9}{baseline[name]:>8.1f}{mean:>10.1f} ± {std:<4.1f}{mean - baseline[name]:>+8.1f}")

Type                       Test ex.     SVM      Legal-BERT   Change
Limitation of liability          38    77.6      73.2 ± 1.6     -4.4
Unilateral termination           38    66.7      77.0 ± 1.8    +10.3
Unilateral change                38    63.5      74.2 ± 2.1    +10.7
Content removal                  13    76.9      72.0 ± 5.7     -4.9
Contract by using                23    80.0      83.8 ± 1.6     +3.8
Choice of law                    13    91.7      96.0 ± 0.0     +4.3
Jurisdiction                     16    74.1      96.0 ± 1.6    +21.9
Arbitration                       7    60.0      75.2 ± 4.7    +15.2


Compared with the SVM baseline, Legal-BERT improved six of the eight types, with the largest gains on Jurisdiction (+21.9), Arbitration (+15.2), Unilateral change (+10.7) and Unilateral termination (+10.3), types that are worded in many different ways. The SVM remained better on Limitation of liability (-4.4) and Content removal (-4.9), which use very typical vocabulary. Content removal has only 13 test examples, so its score varies more between seeds.



---



## 15. The unseen check
Scores the final model (run A) on the full test set and on only the test sentences that never appear in the training data, for all three seeds. If the scores are almost the same, the 33 overlapping sentences did not inflate the results.

In [6]:
train_texts = set(ds["train"]["text"])
unseen_idx = [i for i, t in enumerate(ds["test"]["text"]) if t not in train_texts]
print("Test sentences never seen in training:", len(unseen_idx), "of", len(y_true), "\n")

full, unseen = [], []
for seed in [42, 1, 2]:
    logits = np.load(f"{save_dir}/predictions/test_logits_A_seed{seed}.npy")
    pred = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)
    full.append(lexglue_scores(y_true, pred))
    unseen.append(lexglue_scores(y_true[unseen_idx], pred[unseen_idx]))
    print(f"Seed {seed}: full test micro {full[-1][0]:.1f} / macro {full[-1][1]:.1f} | "
          f"unseen only micro {unseen[-1][0]:.1f} / macro {unseen[-1][1]:.1f}")

full, unseen = np.array(full), np.array(unseen)
print(f"\nAverage | full test: micro {full[:, 0].mean():.1f}, macro {full[:, 1].mean():.1f} | "
      f"unseen only: micro {unseen[:, 0].mean():.1f}, macro {unseen[:, 1].mean():.1f}")

Test sentences never seen in training: 1574 of 1607 

Seed 42: full test micro 96.0 / macro 83.8 | unseen only micro 95.9 / macro 83.7
Seed 1: full test micro 95.6 / macro 81.3 | unseen only micro 95.6 / macro 81.3
Seed 2: full test micro 95.9 / macro 83.5 | unseen only micro 95.8 / macro 83.4

Average | full test: micro 95.8, macro 82.8 | unseen only: micro 95.7, macro 82.8


Scoring the final model on only the 1,574 test sentences that never appear in training gave the same results as the full test set (micro F1 95.7 vs 95.8, macro F1 82.8 vs 82.8, averaged over three seeds). The 33 overlapping sentences therefore did not inflate the results.



---



## 16. Real mistakes
Shows examples of the final model's mistakes on Limitation of liability (the type where the SVM baseline did better): clauses it missed and false alarms, with the model's confidence and the expert labels. This helps explain why some clauses are hard.

In [7]:
logits = np.load(f"{save_dir}/predictions/test_logits_A_seed42.npy")
probs = 1 / (1 + np.exp(-logits))
pred = (probs > 0.5).astype(int)
texts = ds["test"]["text"]
t = names.index("Limitation of liability")

def labels_of(row):
    return [names[k] for k in np.where(row == 1)[0]] or "fair"

missed = [i for i in range(len(y_true)) if y_true[i, t] == 1 and pred[i, t] == 0]
false_alarms = [i for i in range(len(y_true)) if y_true[i, t] == 0 and pred[i, t] == 1]
print(f"Limitation of liability: {len(missed)} missed, {len(false_alarms)} false alarms\n")

print("===== MISSED (should have been flagged) =====")
for i in missed[:3]:
    print(f"Confidence {probs[i, t]:.2f} | expert labels: {labels_of(y_true[i])}")
    print(texts[i][:250], "\n")

print("===== FALSE ALARMS (flagged, but experts said no) =====")
for i in false_alarms[:3]:
    print(f"Confidence {probs[i, t]:.2f} | expert labels: {labels_of(y_true[i])}")
    print(texts[i][:250], "\n")

Limitation of liability: 9 missed, 13 false alarms

===== MISSED (should have been flagged) =====
Confidence 0.00 | expert labels: ['Limitation of liability']
you agree that we are not responsible for others ' ( including other members ' ) content or information . 
 

Confidence 0.00 | expert labels: ['Limitation of liability']
we can not always prevent this misuse of our services , and you agree that we are not responsible for any such misuse . 
 

Confidence 0.00 | expert labels: ['Limitation of liability']
except to the limited extent it may be required by applicable law , linkedin is not responsible for these other sites and apps -- use these at your own risk . 
 

===== FALSE ALARMS (flagged, but experts said no) =====
Confidence 0.99 | expert labels: fair
you acknowledge and agree that academia.edu is not responsible or liable for : ( i ) the availability or accuracy of such websites or resources ; or ( ii ) the content , products , or services on or available from such websites 

On Limitation of liability, the model missed 9 clauses and raised 13 false alarms (seed 42). Some errors involve sentences that are almost identical but were labeled differently by the experts, for example, disclaiming responsibility for third-party websites was labeled unfair in one contract and fair in another. This shows that the boundary between fair and unfair is a subtle legal judgment, and the annotations may not always be consistent. The model was also very confident in several wrong predictions, which supports presenting results as ‘flagged for review’ rather than final decisions.